# Reference-ligand redocking validation -- Aldose reductase (2FZD), AutoDock Vina

Standalone notebook. Validates the docking protocol used for compound screening against **Aldose reductase (PDB 2FZD)** by
redocking the co-crystallized ligand **TOL (tolrestat)** and comparing the docked poses with the crystal pose.

**Protocol identical to the screening notebooks** (`1_-Molecular_Docking_2FZD_*.ipynb`): same receptor
(`2FZD_clean_H.pdbqt`, LePro + AutoDockTools `prepare_receptor`), same box (native ligand + 5.0 Å, Jupyter_Dock `getbox`),
AutoDock Vina 1.2 with the `vina` scoring function, exhaustiveness 10, 10 poses, seed 42.

**RMSD is computed in place** (`rdMolAlign.CalcRMS`: heavy atoms, symmetry-aware, *no superposition*), i.e. in the
binding-site frame, which is the standard redocking criterion (success: RMSD < 2.0 Å). The earlier redocking cell in the
screening notebooks used `GetBestRMS`, which first superimposes the pose onto the crystal ligand and therefore ignores where
the pose is in the site; that value is still reported here, for comparison only. Bond orders are assigned from the
ligand's SMILES template, and an element-based MCS in-place RMSD is used as fallback if the template match fails
(this is what made the RMSD uncomputable for 2FZD before).

**Contents:** 1. Setup -- 2. Fetch & clean (skipped if files exist) -- 3. Receptor -- 4. Docking box -- 5. Native ligand --
6. Redocking -- 7. RMSD vs crystal pose -- 8. Seed robustness -- 9. Result for Table 15-- 10. NADP⁺ cofactor sensitivity (reviewer comment 4)

In [1]:
import os, sys, glob, shutil, subprocess
import numpy as np, pandas as pd
from pymol import cmd
from vina import Vina
from openbabel import openbabel as ob
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolAlign, rdFMCS
import warnings
warnings.filterwarnings("ignore"); RDLogger.DisableLog("rdApp.*")

# ---- Jupyter_Dock utilities (same getbox as the screening notebooks) ----
JD = "/home/ali/medical plants/docking/Jupyter_Dock-main"
sys.path.insert(1, os.path.join(JD, "utilities"))
try:
    from utils import getbox
    print("Using Jupyter_Dock getbox()")
except ImportError:                       # identical arithmetic, used only if utils is not on the path
    def getbox(selection="sele", extending=6.0, software="vina"):
        ([minX, minY, minZ], [maxX, maxY, maxZ]) = cmd.get_extent(selection)
        minX, minY, minZ = minX - extending, minY - extending, minZ - extending
        maxX, maxY, maxZ = maxX + extending, maxY + extending, maxZ + extending
        size   = {"size_x": maxX - minX, "size_y": maxY - minY, "size_z": maxZ - minZ}
        center = {"center_x": (maxX + minX) / 2, "center_y": (maxY + minY) / 2, "center_z": (maxZ + minZ) / 2}
        return center, size
    print("Jupyter_Dock utils not found -- using built-in getbox() (same formula)")

LEPRO_BIN    = os.path.join(JD, "bin", "lepro_linux_x86")
PREP_REC_BIN = os.path.join(JD, "bin", "prepare_receptor")
PREP_LIG_BIN = os.path.join(JD, "bin", "prepare_ligand")
for b in (LEPRO_BIN, PREP_REC_BIN, PREP_LIG_BIN):
    print(("found    " if os.path.exists(b) else "MISSING  ") + b)

Using Jupyter_Dock getbox()
found    /home/ali/medical plants/docking/Jupyter_Dock-main/bin/lepro_linux_x86
found    /home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_receptor
found    /home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_ligand


In [2]:
# ---------------- configuration ----------------
DOCKING_DIR = "/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/"   # same folder as screening
PDB_CODE    = "2FZD"
TARGET      = "Aldose reductase"
NATIVE_SEL  = "resn TOL and chain A"           # co-crystallized ligand (TOL (tolrestat))
NATIVE_SMILES = "CN(CC(=O)O)C(=S)c1cccc2c1ccc(OC)c2C(F)(F)F"   # bond-order template

VINA_SEED, EXHAUSTIVENESS, N_POSES = 42, 10, 10        # identical to screening
BOX_EXTENDING = 5.0
RMSD_CUTOFF   = 2.0
ADD_HYDROGENS = True     # protonate the crystal ligand at pH 7.4 (screening ligands carry H too)
SEEDS_CHECK   = [42, 1, 7, 123, 2024]                  # section 8

# Box reported in the paper (Table 16) -- used only to check that the same box is recomputed
PAPER_CENTER = (16.6905, -7.611, 13.869)
PAPER_SIZE   = (13.607, 15.78, 19.872)
PAPER_RMSD   = "0.307 Å (Table 15, value from the earlier GetBestRMS cell)"

os.makedirs(DOCKING_DIR, exist_ok=True)
REDOCK_DIR = os.path.join(DOCKING_DIR, "redocking_validation", PDB_CODE)
os.makedirs(REDOCK_DIR, exist_ok=True)
os.chdir(DOCKING_DIR)

CLEAN_PDB      = os.path.join(DOCKING_DIR, f"{PDB_CODE}_clean.pdb")
RECEPTOR_PDB   = os.path.join(DOCKING_DIR, f"{PDB_CODE}_clean_H.pdb")
RECEPTOR_PDBQT = os.path.join(DOCKING_DIR, f"{PDB_CODE}_clean_H.pdbqt")
NATIVE_LIG     = os.path.join(DOCKING_DIR, f"{PDB_CODE}_native_lig.mol2")
print("Working folder :", DOCKING_DIR)
print("Redock outputs :", REDOCK_DIR)

Working folder : /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/
Redock outputs : /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/redocking_validation/2FZD


## 2. Fetch and clean the structure

Runs only if the files from the screening notebooks are missing, so the validated receptor is reused unchanged.

In [3]:
if not (os.path.exists(CLEAN_PDB) and os.path.exists(NATIVE_LIG)):
    print(f"Fetching {PDB_CODE} ...")
    cmd.fetch(code=PDB_CODE, type="pdb1")
    cmd.select(name="Prot", selection="polymer.protein")
    cmd.save(filename=CLEAN_PDB, format="pdb", selection="Prot", state=1)
    cmd.select(name="NativeLig", selection=NATIVE_SEL)
    cmd.save(filename=NATIVE_LIG, format="mol2", selection="NativeLig", state=1)
    cmd.delete("all")
    for f in glob.glob(f"{PDB_CODE.lower()}*.pdb*") + glob.glob(f"{PDB_CODE}*.cif"):
        if os.path.abspath(f) not in (CLEAN_PDB,): os.remove(f)     # remove the raw download
else:
    print("Reusing existing files from the screening notebooks")

cmd.load(NATIVE_LIG, "natlig")
_m = cmd.get_model("natlig")
print(f"Native ligand: {sorted(set((a.resn, a.resi, a.chain) for a in _m.atom))}, "
      f"{sum(a.symbol != 'H' for a in _m.atom)} heavy atoms "
      f"(template {Chem.MolFromSmiles(NATIVE_SMILES).GetNumHeavyAtoms()})")
cmd.delete("all")

Reusing existing files from the screening notebooks
Native ligand: [('TOL', '317', 'A')], 24 heavy atoms (template 24)


## 3. Receptor (same preparation as screening: LePro → `prepare_receptor`)

In [4]:
def run(cmdline):
    r = subprocess.run(cmdline, capture_output=True, text=True, shell=isinstance(cmdline, str))
    return r.returncode, (r.stdout + r.stderr)[-600:]

if not os.path.exists(RECEPTOR_PDBQT):
    if not os.path.exists(RECEPTOR_PDB):
        if os.path.exists(LEPRO_BIN):
            print("LePro ..."); run([LEPRO_BIN, CLEAN_PDB]); os.rename("pro.pdb", RECEPTOR_PDB)
        else:
            print("LePro not found -- adding hydrogens with OpenBabel at pH 7.4 (fallback)")
            run(["obabel", CLEAN_PDB, "-O", RECEPTOR_PDB, "-h", "-p", "7.4"])
    if os.path.exists(PREP_REC_BIN):
        print("prepare_receptor ..."); run([PREP_REC_BIN, "-v", "-r", RECEPTOR_PDB, "-o", RECEPTOR_PDBQT])
    if not os.path.exists(RECEPTOR_PDBQT) or os.path.getsize(RECEPTOR_PDBQT) == 0:
        print("prepare_receptor unavailable/failed -- OpenBabel -xr fallback")
        run(["obabel", RECEPTOR_PDB, "-O", RECEPTOR_PDBQT, "-xr"])
else:
    print("Reusing", RECEPTOR_PDBQT)
assert os.path.getsize(RECEPTOR_PDBQT) > 0
n_atoms = sum(1 for l in open(RECEPTOR_PDBQT) if l.startswith(("ATOM", "HETATM")))
print(f"Receptor PDBQT: {n_atoms} atoms")

Reusing /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/2FZD_clean_H.pdbqt
Receptor PDBQT: 3038 atoms


## 4. Docking box (native ligand + 5.0 Å) -- checked against Table 16

In [5]:
cmd.load(RECEPTOR_PDB, "prot"); cmd.load(NATIVE_LIG, "natlig")
center, size = getbox(selection="natlig", extending=BOX_EXTENDING, software="vina")
cmd.delete("all")
C = [center["center_x"], center["center_y"], center["center_z"]]
S = [size["size_x"], size["size_y"], size["size_z"]]
print("center:", [round(v, 4) for v in C], "\nsize  :", [round(v, 4) for v in S])
dc = max(abs(a - b) for a, b in zip(C, PAPER_CENTER)); ds = max(abs(a - b) for a, b in zip(S, PAPER_SIZE))
print(f"Max difference from Table 16: centre {dc:.4f} Å, size {ds:.4f} Å ->",
      "same box as the paper" if dc < 0.01 and ds < 0.01 else "CHECK: box differs from Table 16")

center: [16.6905, -7.611, 13.869] 
size  : [13.607, 15.78, 19.872]
Max difference from Table 16: centre 0.0000 Å, size 0.0000 Å -> same box as the paper


## 5. Native ligand (crystal geometry → PDBQT)

In [6]:
def obconvert(src, dst, add_h_ph=None):
    conv = ob.OBConversion()
    conv.SetInAndOutFormats(os.path.splitext(src)[1][1:], os.path.splitext(dst)[1][1:])
    mol = ob.OBMol(); conv.ReadFile(mol, src)
    if add_h_ph is not None:
        mol.CorrectForPH(add_h_ph); mol.AddHydrogens(False, True, add_h_ph)
    conv.WriteFile(mol, dst)
    return mol.NumAtoms()

native_h_pdb = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_H.pdb")
native_pdbqt = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_redock.pdbqt")
n = obconvert(NATIVE_LIG, native_h_pdb, add_h_ph=7.4 if ADD_HYDROGENS else None)
print(f"Native ligand with hydrogens: {n} atoms ({'pH 7.4' if ADD_HYDROGENS else 'no H added'})")

if os.path.exists(PREP_LIG_BIN):
    run([PREP_LIG_BIN, "-v", "-l", native_h_pdb, "-o", native_pdbqt])
if not os.path.exists(native_pdbqt) or os.path.getsize(native_pdbqt) == 0:
    obconvert(native_h_pdb, native_pdbqt)
tors = sum(1 for l in open(native_pdbqt) if l.startswith("BRANCH"))
print(f"Ligand PDBQT ready: {native_pdbqt}  ({tors} rotatable branches)")

Native ligand with hydrogens: 38 atoms (pH 7.4)
Ligand PDBQT ready: /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/redocking_validation/2FZD/2FZD_native_lig_redock.pdbqt  (6 rotatable branches)


## 6. Redocking (AutoDock Vina, screening settings)

In [7]:
def vina_dock(receptor, ligand, out, seed=VINA_SEED, exh=EXHAUSTIVENESS, n_poses=N_POSES):
    v = Vina(sf_name="vina", seed=seed, verbosity=0)
    v.set_receptor(receptor); v.set_ligand_from_file(ligand)
    v.compute_vina_maps(center=C, box_size=S)
    v.dock(exhaustiveness=exh, n_poses=n_poses)
    v.write_poses(out, n_poses=n_poses, overwrite=True)
    return [float(e[0]) for e in v.energies(n_poses=n_poses)]

redock_out = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_redock_out.pdbqt")
print(f"Redocking {NATIVE_SEL} into {PDB_CODE} (seed {VINA_SEED}, exhaustiveness {EXHAUSTIVENESS}) ...")
scores = vina_dock(RECEPTOR_PDBQT, native_pdbqt, redock_out)
print("Vina scores (kcal/mol):", [round(s, 3) for s in scores])

Redocking resn TOL and chain A into 2FZD (seed 42, exhaustiveness 10) ...
Vina scores (kcal/mol): [-9.023, -7.643, -7.514, -7.5, -7.366, -7.362, -7.352, -7.217, -6.926, -6.75]


## 7. RMSD vs crystal pose (in place, heavy atoms, symmetry-aware)

In [8]:
template = Chem.MolFromSmiles(NATIVE_SMILES)

def with_bond_orders(pdb_block):
    """Heavy-atom RDKit molecule from a PDB block, bond orders assigned from the SMILES template."""
    m = Chem.MolFromPDBBlock(pdb_block, removeHs=False, sanitize=False, proximityBonding=True)
    m = Chem.RemoveHs(m, sanitize=False)
    try:
        return AllChem.AssignBondOrdersFromTemplate(template, m), True
    except Exception:
        return m, False

def pdb_blocks(path):
    """One PDB block per structure in a mol2/pdb/pdbqt file (OpenBabel)."""
    conv = ob.OBConversion(); conv.SetInAndOutFormats(os.path.splitext(path)[1][1:], "pdb")
    mol = ob.OBMol(); blocks = []
    ok = conv.ReadFile(mol, path)
    while ok:
        mol.DeleteHydrogens(); blocks.append(conv.WriteString(mol))
        mol = ob.OBMol(); ok = conv.Read(mol)
    return blocks

def mcs_inplace_rmsd(ref, probe):
    """Fallback: element-based MCS, all symmetry-equivalent matches, in place (no superposition)."""
    mcs = rdFMCS.FindMCS([ref, probe], atomCompare=rdFMCS.AtomCompare.CompareElements,
                         bondCompare=rdFMCS.BondCompare.CompareAny, ringMatchesRingOnly=False, timeout=10)
    patt = Chem.MolFromSmarts(mcs.smartsString)
    rm = ref.GetSubstructMatches(patt, uniquify=False, maxMatches=1000)
    pm = probe.GetSubstructMatch(patt)
    if not rm or not pm: return None, 0
    rc, pc = ref.GetConformer().GetPositions(), probe.GetConformer().GetPositions()
    best = min(np.sqrt(np.mean(np.sum((rc[list(r)] - pc[list(pm)]) ** 2, axis=1))) for r in rm)
    return round(float(best), 3), len(pm)

def rmsd_table(pose_file, crystal_file, energies):
    ref, ref_ok = with_bond_orders(pdb_blocks(crystal_file)[0])
    rows = []
    for i, blk in enumerate(pdb_blocks(pose_file)):
        pose, ok = with_bond_orders(blk)
        r_inplace = r_super = None; method = "template"
        if ok and ref_ok:
            try:
                r_inplace = round(rdMolAlign.CalcRMS(Chem.Mol(pose), ref), 3)      # in place
                r_super   = round(rdMolAlign.GetBestRMS(Chem.Mol(pose), ref), 3)   # superimposed (old method)
            except Exception:
                ok = False
        if r_inplace is None:
            r_inplace, n = mcs_inplace_rmsd(ref, pose); method = f"MCS ({n} atoms)"
        rows.append({"Pose": i + 1, "Vina_score_kcal_mol": round(energies[i], 3),
                     "RMSD_in_place_A": r_inplace, "RMSD_superimposed_A": r_super, "Atom_matching": method})
    return pd.DataFrame(rows)

redock_df = rmsd_table(redock_out, NATIVE_LIG, scores)
display(redock_df)

top = redock_df.iloc[0]                         # best-scoring pose = the pose the protocol would report
best_any = redock_df["RMSD_in_place_A"].min()
RESULT = "PASS" if top["RMSD_in_place_A"] is not None and top["RMSD_in_place_A"] < RMSD_CUTOFF else "FAIL"
print(f"Top-scoring pose: score {top['Vina_score_kcal_mol']} kcal/mol, "
      f"RMSD in place {top['RMSD_in_place_A']} Å (superimposed: {top['RMSD_superimposed_A']} Å)")
print(f"Lowest in-place RMSD among the {len(redock_df)} poses: {best_any} Å")
print(f"Redocking validation (top pose, in place, < {RMSD_CUTOFF} Å): {RESULT}")
print(f"Previously reported in the paper: {PAPER_RMSD}")
redock_df.to_csv(os.path.join(REDOCK_DIR, f"{PDB_CODE}_redocking_validation.csv"), index=False)

,Pose,Vina_score_kcal_mol,RMSD_in_place_A,RMSD_superimposed_A,Atom_matching
0,1,-9.023,0.390,0.304,template
1,2,-7.643,3.375,1.213,template
2,3,-7.514,6.387,0.440,template
3,4,-7.500,3.426,1.663,template
4,5,-7.366,6.817,0.609,template
5,6,-7.362,6.118,1.197,template
6,7,-7.352,5.689,0.513,template
7,8,-7.217,2.520,1.234,template
8,9,-6.926,6.414,1.689,template
9,10,-6.750,7.504,1.219,template


Top-scoring pose: score -9.023 kcal/mol, RMSD in place 0.39 Å (superimposed: 0.304 Å)
Lowest in-place RMSD among the 10 poses: 0.39 Å
Redocking validation (top pose, in place, < 2.0 Å): PASS
Previously reported in the paper: 0.307 Å (Table 15, value from the earlier GetBestRMS cell)


## 8. Seed robustness

The same redocking with different random seeds; a validated protocol should reproduce the crystal pose for most seeds.

In [9]:
seed_rows = []
for sd in SEEDS_CHECK:
    out = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_redock_seed{sd}.pdbqt")
    en = vina_dock(RECEPTOR_PDBQT, native_pdbqt, out, seed=sd)
    d = rmsd_table(out, NATIVE_LIG, en)
    seed_rows.append({"Seed": sd, "Top_score_kcal_mol": d.iloc[0]["Vina_score_kcal_mol"],
                      "Top_pose_RMSD_in_place_A": d.iloc[0]["RMSD_in_place_A"],
                      "Lowest_RMSD_any_pose_A": d["RMSD_in_place_A"].min()})
    print(f"  seed {sd:5d}: top score {seed_rows[-1]['Top_score_kcal_mol']:.3f}, "
          f"top-pose RMSD {seed_rows[-1]['Top_pose_RMSD_in_place_A']} Å")
seed_df = pd.DataFrame(seed_rows)
n_pass = int((seed_df["Top_pose_RMSD_in_place_A"] < RMSD_CUTOFF).sum())
print(f"\nTop pose within {RMSD_CUTOFF} Å for {n_pass} of {len(seed_df)} seeds")
seed_df.to_csv(os.path.join(REDOCK_DIR, f"{PDB_CODE}_redocking_seed_robustness.csv"), index=False)
display(seed_df)

  seed    42: top score -9.023, top-pose RMSD 0.39 Å


  seed     1: top score -9.036, top-pose RMSD 0.416 Å


  seed     7: top score -9.010, top-pose RMSD 0.398 Å


  seed   123: top score -9.018, top-pose RMSD 0.412 Å
  seed  2024: top score -9.018, top-pose RMSD 0.387 Å

Top pose within 2.0 Å for 5 of 5 seeds


,Seed,Top_score_kcal_mol,Top_pose_RMSD_in_place_A,Lowest_RMSD_any_pose_A
0,42,-9.023,0.390,0.390
1,1,-9.036,0.416,0.416
2,7,-9.010,0.398,0.398
3,123,-9.018,0.412,0.412
4,2024,-9.018,0.387,0.387


## 9. Result for Table 15

In [10]:
summary = {"Target (PDB)": f"{TARGET} ({PDB_CODE})", "Native ligand": NATIVE_SEL, "Engine": "AutoDock Vina 1.2",
           "Top-pose score (kcal/mol)": top["Vina_score_kcal_mol"],
           "RMSD in place (Å)": top["RMSD_in_place_A"], "RMSD superimposed (Å)": top["RMSD_superimposed_A"],
           "Seeds passing": f"{n_pass}/{len(seed_df)}", "Result": RESULT}
pd.DataFrame([summary]).to_csv(os.path.join(REDOCK_DIR, f"{PDB_CODE}_table15_row.csv"), index=False)
for k, v in summary.items(): print(f"{k:28s} {v}")


Target (PDB)                 Aldose reductase (2FZD)
Native ligand                resn TOL and chain A
Engine                       AutoDock Vina 1.2
Top-pose score (kcal/mol)    -9.023
RMSD in place (Å)            0.39
RMSD superimposed (Å)        0.304
Seeds passing                5/5
Result                       PASS

If 'RMSD in place' differs from the value in Table 15, update Table 15 with the in-place value.


## 10. NADP⁺ cofactor sensitivity 

In 2FZD the nicotinamide of NADP⁺ forms part of the inhibitor pocket (closest NADP⁺ heavy atom 3.03 Å from tolrestat), but the
screening receptor contains no NADP⁺. Here a second receptor is built from the **same validated protein PDBQT** plus NADP⁺ (residue `NAP`, chain A, crystal coordinates; bond orders from a SMILES template, ionised at pH 7.4, rigid), and both the
tolrestat redocking and the 20 PHYTO compounds are docked against **both** receptors with identical settings and the same box.
Both runs are in this notebook, so the comparison does not depend on earlier files.

Outputs: redocking RMSD with NADP⁺, per-compound score change (with − without), and the Spearman correlation of the two
rankings.

In [11]:
REC_NADP_PDBQT = os.path.join(REDOCK_DIR, "2FZD_with_NADP.pdbqt")
NAP_PDB        = os.path.join(REDOCK_DIR, "2FZD_NADP.pdb")
NAP_SDF        = os.path.join(REDOCK_DIR, "2FZD_NADP_H.sdf")
NAP_PDBQT      = os.path.join(REDOCK_DIR, "2FZD_NADP_H.pdbqt")

# 1) NADP+ (chain A) crystal coordinates from the same biological assembly
cmd.fetch(code=PDB_CODE, type="pdb1"); cmd.select("NAP", "resn NAP and chain A")
print("NADP+ atoms in crystal (chain A):", cmd.count_atoms("NAP"))
cmd.save(NAP_PDB, "NAP", state=1); cmd.delete("all")
for f in glob.glob(f"{PDB_CODE.lower()}*.pdb*") + glob.glob(f"{PDB_CODE}*.cif"): os.remove(f)

# 2) Bond orders from a SMILES template (OpenBabel cannot perceive NADP+'s aromatic rings from a PDB),
#    ionisation at pH 7.4 (phosphates deprotonated, nicotinamide N+), hydrogens added on the crystal frame
NADP_TEMPLATE = ("NC(=O)c1ccc[n+](c1)C1OC(COP(=O)([O-])OP(=O)([O-])OCC2OC(n3cnc4c(N)ncnc43)"
                 "C(OP(=O)([O-])[O-])C2O)C(O)C1O")
tmpl = Chem.MolFromSmiles(NADP_TEMPLATE)
nap  = Chem.MolFromPDBFile(NAP_PDB, removeHs=True, sanitize=False, proximityBonding=True)
nap  = AllChem.AssignBondOrdersFromTemplate(tmpl, nap)
nap_h = Chem.AddHs(nap, addCoords=True)
print(f"NADP+ prepared: {nap.GetNumAtoms()} heavy atoms (template {tmpl.GetNumAtoms()}), "
      f"{nap_h.GetNumAtoms()} atoms with H, net charge {Chem.GetFormalCharge(nap_h)}")
Chem.MolToMolFile(nap_h, NAP_SDF)
heavy_shift = max(abs(a - b) for a, b in zip(nap_h.GetConformer().GetPositions()[:nap.GetNumAtoms()].ravel(),
                                               nap.GetConformer().GetPositions().ravel()))
print(f"Heavy atoms kept at crystal coordinates (max shift {heavy_shift:.3f} Å)")

# 3) Rigid PDBQT for NADP+ (AutoDock atom types from the correct bond orders), labelled as residue NAP
run(["obabel", NAP_SDF, "-O", NAP_PDBQT, "-xr"])
nap_lines = []
for l in open(NAP_PDBQT):
    if l.startswith(("ATOM", "HETATM")):
        l = "HETATM" + l[6:17] + "NAP" + " A" + " 317" + l[26:]   # resname NAP, chain A, resi 317
        nap_lines.append(l)
types = sorted(set(l[77:79].strip() for l in nap_lines))
print(f"NADP+ PDBQT: {len(nap_lines)} atoms, AutoDock types {types}")

# 4) Receptor with cofactor = the SAME validated protein PDBQT used above + NADP+
prot_lines = [l for l in open(RECEPTOR_PDBQT) if l.startswith(("ATOM", "HETATM"))]
with open(REC_NADP_PDBQT, "w") as f:
    f.writelines(prot_lines); f.write("TER\n"); f.writelines(nap_lines); f.write("END\n")
nap_in  = sum(1 for l in open(REC_NADP_PDBQT) if l.startswith(("ATOM", "HETATM")) and l[17:20] == "NAP")
in_box  = sum(1 for l in nap_lines if all(abs(float(l[30 + 8 * k:38 + 8 * k]) - C[k]) <= S[k] / 2 for k in range(3)))
print(f"Receptor with NADP+: {len(prot_lines)} protein + {nap_in} NADP+ atoms ({in_box} NADP+ atoms inside the docking box)")
assert nap_in == len(nap_lines) > 0, "NADP+ missing from the combined receptor"

# 5) Tolrestat redocking with the cofactor present
redock_nadp_out = os.path.join(REDOCK_DIR, "2FZD_native_lig_redock_withNADP_out.pdbqt")
scores_nadp = vina_dock(REC_NADP_PDBQT, native_pdbqt, redock_nadp_out)
redock_nadp_df = rmsd_table(redock_nadp_out, NATIVE_LIG, scores_nadp)
top_n = redock_nadp_df.iloc[0]
print(f"Tolrestat redocking WITH NADP+   : score {top_n['Vina_score_kcal_mol']} kcal/mol, "
      f"RMSD in place {top_n['RMSD_in_place_A']} Å -> "
      f"{'PASS' if top_n['RMSD_in_place_A'] is not None and top_n['RMSD_in_place_A'] < RMSD_CUTOFF else 'FAIL'}")
print(f"Tolrestat redocking WITHOUT NADP+: score {top['Vina_score_kcal_mol']} kcal/mol, RMSD in place {top['RMSD_in_place_A']} Å")
redock_nadp_df.to_csv(os.path.join(REDOCK_DIR, "2FZD_redocking_validation_withNADP.csv"), index=False)


NADP+ atoms in crystal (chain A): 48
NADP+ prepared: 48 heavy atoms (template 48), 73 atoms with H, net charge -3
Heavy atoms kept at crystal coordinates (max shift 0.000 Å)
NADP+ PDBQT: 55 atoms, AutoDock types ['A', 'C', 'HD', 'N', 'NA', 'OA', 'P']
Receptor with NADP+: 3038 protein + 55 NADP+ atoms (13 NADP+ atoms inside the docking box)
Tolrestat redocking WITH NADP+   : score -8.944 kcal/mol, RMSD in place 0.471 Å -> PASS
Tolrestat redocking WITHOUT NADP+: score -9.023 kcal/mol, RMSD in place 0.39 Å


In [12]:
# 20 PHYTO compounds (same as Table 17). Linalool and cis-dihydrocarvone are the CORRECTED
# structures (the screening notebooks used C11H20O for linalool and an isomeric
# isopropyl-cyclohexenone for cis-dihydrocarvone). Sitosterol and GlcMethylKaempferol were
# docked in notebooks not included here: replace these two SMILES with the exact strings
# used in your screening notebooks if they differ.
COMPOUNDS = {
    "Stigmasterol":        "CC[C@H](/C=C/[C@@H](C)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CC=C4[C@@]3(CC[C@@H](C4)O)C)C)C(C)C",
    "Sitosterol":          "CC[C@H](CC[C@@H](C)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CC=C4[C@@]3(CC[C@@H](C4)O)C)C)C(C)C",
    "GlcMethylKaempferol": "COc1c(-c2ccc(O)cc2)oc2cc(O[C@@H]3O[C@H](CO)[C@@H](O)[C@H](O)[C@H]3O)cc(O)c2c1=O",
    "Chrysoeriol":         "COC1=C(C=CC(=C1)C2=CC(=O)C3=C(C=C(C=C3O2)O)O)O",
    "beta_Caryophyllene":  "CC1=CCCC(=C)C2CC(C2CC1)(C)C",
    "p_Cymene":            "Cc1ccc(C(C)C)cc1",
    "Linalyl_acetate":     "CC(=CCCC(C)(C=C)OC(C)=O)C",
    "Lavandulyl_acetate":  "CC(=CCC(C)(C=C)COC(C)=O)C",
    "Carvone":             "CC1=CCC(CC1=O)C(=C)C",
    "p_Cymen_8_ol":        "CC1=CC=C(C=C1)C(C)(C)O",
    "cis_Dihydrocarvone":  "C[C@@H]1CC[C@@H](CC1=O)C(=C)C",      # corrected
    "Limonene":            "CC1=CCC(CC1)C(=C)C",
    "Linalool":            "CC(C)=CCCC(C)(O)C=C",                 # corrected
    "Cineole_1_8":         "CC1(C2CCC(O1)(CC2)C)C",
    "cis_Verbenol":        "CC1=CC(C2CC1C2(C)C)O",
    "Myrtenal":            "CC1(C)C2CC=C(C=O)C1C2",
    "Camphor":             "CC1(C2CCC1(C(=O)C2)C)C",
    "Fenchone":            "CC1(C)C2CCC1(C)C(=O)C2",
    "alpha_Pinene":        "CC1=CCC2CC1C2(C)C",
    "Hexyl_butyrate":      "CCCCCCOC(=O)CCC",
}

def ligand_from_smiles(name, smi, n_confs=20):
    """Same ligand preparation as the screening notebooks: ETKDGv3 (seed 42), MMFF94 (UFF fallback),
    lowest-energy conformer -> prepare_ligand (OpenBabel fallback)."""
    m = Chem.AddHs(Chem.MolFromSmiles(smi))
    p = AllChem.ETKDGv3(); p.randomSeed = 42
    cids = AllChem.EmbedMultipleConfs(m, numConfs=n_confs, params=p)
    en = []
    for cid in cids:
        ff = AllChem.MMFFGetMoleculeForceField(m, AllChem.MMFFGetMoleculeProperties(m), confId=cid)
        if ff is None: continue
        ff.Minimize(maxIts=2000); en.append((ff.CalcEnergy(), cid))
    if not en:
        for cid in cids:
            ff = AllChem.UFFGetMoleculeForceField(m, confId=cid); ff.Minimize(maxIts=2000); en.append((ff.CalcEnergy(), cid))
    best = min(en)[1]
    pdb_p, pdbqt_p = [os.path.join(REDOCK_DIR, f"_lig_{name}.{x}") for x in ("pdb", "pdbqt")]
    Chem.MolToPDBFile(m, pdb_p, confId=best)
    if os.path.exists(PREP_LIG_BIN): run([PREP_LIG_BIN, "-v", "-l", pdb_p, "-o", pdbqt_p])
    if not os.path.exists(pdbqt_p) or os.path.getsize(pdbqt_p) == 0: obconvert(pdb_p, pdbqt_p)
    return pdbqt_p

rows = []
for k, (name, smi) in enumerate(COMPOUNDS.items(), 1):
    lig = ligand_from_smiles(name, smi)
    s0 = vina_dock(RECEPTOR_PDBQT, lig, os.path.join(REDOCK_DIR, f"{name}_2FZD_noNADP_out.pdbqt"))[0]
    s1 = vina_dock(REC_NADP_PDBQT, lig, os.path.join(REDOCK_DIR, f"{name}_2FZD_withNADP_out.pdbqt"))[0]
    rows.append({"Compound": name, "Score_without_NADP": round(s0, 3), "Score_with_NADP": round(s1, 3),
                 "Delta_with_minus_without": round(s1 - s0, 3)})
    print(f"[{k:2d}/{len(COMPOUNDS)}] {name:20s} without {s0:7.3f} | with NADP+ {s1:7.3f} | delta {s1 - s0:+.3f}")

cof = pd.DataFrame(rows)
cof["Rank_without"] = cof["Score_without_NADP"].rank(method="min").astype(int)
cof["Rank_with"]    = cof["Score_with_NADP"].rank(method="min").astype(int)
rho = cof["Score_without_NADP"].corr(cof["Score_with_NADP"], method="spearman")
cof = cof.sort_values("Score_without_NADP").reset_index(drop=True)
cof.to_csv(os.path.join(REDOCK_DIR, "2FZD_cofactor_sensitivity.csv"), index=False)
display(cof)
print(f"\nMean change with NADP+: {cof['Delta_with_minus_without'].mean():+.3f} kcal/mol "
      f"(range {cof['Delta_with_minus_without'].min():+.3f} to {cof['Delta_with_minus_without'].max():+.3f})")
print(f"Spearman rank correlation (without vs with NADP+): {rho:.3f}")
print("Top 5 without NADP+:", cof.nsmallest(5, 'Score_without_NADP')['Compound'].tolist())
print("Top 5 with NADP+   :", cof.nsmallest(5, 'Score_with_NADP')['Compound'].tolist())

[ 1/20] Stigmasterol         without  -9.033 | with NADP+  -8.397 | delta +0.636


[ 2/20] Sitosterol           without  -9.900 | with NADP+  -8.284 | delta +1.616


[ 3/20] GlcMethylKaempferol  without  -8.373 | with NADP+  -8.364 | delta +0.009


[ 4/20] Chrysoeriol          without  -8.276 | with NADP+  -7.838 | delta +0.438


[ 5/20] beta_Caryophyllene   without  -7.992 | with NADP+  -8.149 | delta -0.157


...

[18/20] Fenchone             without  -5.832 | with NADP+  -6.293 | delta -0.461


[19/20] alpha_Pinene         without  -6.248 | with NADP+  -6.849 | delta -0.601


[20/20] Hexyl_butyrate       without  -5.845 | with NADP+  -5.881 | delta -0.036


,Compound,Score_without_NADP,Score_with_NADP,Delta_with_minus_without,Rank_without,Rank_with
0,Sitosterol,-9.900,-8.284,1.616,1,3
1,Stigmasterol,-9.033,-8.397,0.636,2,1
2,GlcMethylKaempferol,-8.373,-8.364,0.009,3,2
3,Chrysoeriol,-8.276,-7.838,0.438,4,5
4,beta_Caryophyllene,-7.992,-8.149,-0.157,5,4
5,Lavandulyl_acetate,-6.826,-6.660,0.166,6,13
6,Carvone,-6.609,-6.730,-0.121,7,9
7,Linalyl_acetate,-6.550,-6.934,-0.384,8,6
8,cis_Dihydrocarvone,-6.483,-6.924,-0.441,9,7
9,Linalool,-6.417,-6.696,-0.279,10,11



Mean change with NADP+: -0.092 kcal/mol (range -0.601 to +1.616)
Spearman rank correlation (without vs with NADP+): 0.838
Top 5 without NADP+: ['Sitosterol', 'Stigmasterol', 'GlcMethylKaempferol', 'Chrysoeriol', 'beta_Caryophyllene']
Top 5 with NADP+   : ['Stigmasterol', 'GlcMethylKaempferol', 'Sitosterol', 'beta_Caryophyllene', 'Chrysoeriol']
